# Clustering des séries temporelles de consommation énergétique des bâtiments individuels

## Objectif du notebook

L'objectif de ce notebook est d'analyser les séries temporelles de consommation énergétique des bâtiments individuels afin d'identifier des **jours types de consommation**.

L'idée est de regrouper les journées présentant des comportements similaires en appliquant une méthode de **clustering non supervisé**. Ces profils journaliers caractéristiques permettront ensuite de mieux comprendre les habitudes de consommation des bâtiments et d'identifier différents comportements énergétiques.

---

## Approche suivie

La méthodologie est organisée en plusieurs étapes :

1. **Chargement des séries temporelles**
   - Lecture des fichiers de consommation énergétique des bâtiments individuels.
   - Sélection de la variable de consommation électrique étudiée.
   - Vérification de la fréquence temporelle des données.

2. **Transformation des séries temporelles en profils journaliers**
   - Les séries annuelles (pas de temps de 15 minutes) sont restructurées sous forme de matrices :
   Chaque ligne représente alors un profil de consommation sur une journée.

3. **Prétraitement des profils**
   - Normalisation des profils journaliers afin de comparer les formes de consommation indépendamment du niveau énergétique absolu.
   - Cette étape permet de détecter des comportements similaires même lorsque les bâtiments ont des consommations différentes.

4. **Détermination du nombre optimal de clusters**
   - Plusieurs valeurs du nombre de groupes sont testées.
   - Le score de silhouette est utilisé pour mesurer la qualité de séparation des clusters.

5. **Application du clustering**
   - Utilisation de l'algorithme **K-Means** pour regrouper les jours présentant des profils similaires.
   - Chaque cluster représente un **jour type de consommation**.

6. **Analyse et visualisation des résultats**
   - Visualisation des centroïdes des clusters correspondant aux journées représentatives.
   - Analyse de la fréquence d'apparition de chaque type de journée.


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import glob
import time
import joblib

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
from kneed import KneeLocator
import seaborn as sns

ROOT = Path().resolve().parent.parent
DATA_RAW       = ROOT / 'data' / 'raw'
DATA_RAW_100   = DATA_RAW / 'timeseries_100_all_electric'  # séries des 100 bâtiments tout-électriques
DATA_PROCESSED = ROOT / 'data' / 'processed'
FIGURES        = ROOT / 'reports' / 'figures'

COL = "out.electricity.total.energy_consumption..kwh"

# Pas de temps journalier : 24 points (heures) au lieu de 96 (15 min)
RESAMPLE_FREQ  = "h"
steps_per_day  = 24
HOURS          = np.arange(steps_per_day)

WEATHER_COLS = [
    "out.outdoor_air_drybulb_temp..c",
    "out.outdoor_air_relative_humidity..percentage",
    "out.outdoor_air_wetbulb_temp..c",
    "out.outdoor_humidity_ratio..kgwater_per_kgdryair",
]

## Exploration rapide d'un bâtiment (sanity check visuel)

In [ ]:
df = pd.read_parquet(DATA_RAW / "347201-0.parquet")
df.head()

In [ ]:
dff = (
    pd.read_parquet(DATA_RAW / "347201-0.parquet")
    .assign(timestamp=lambda x: x["timestamp"] - pd.Timedelta("15m"))
    .set_index("timestamp")
    .loc[:, lambda x: x.columns.str.match(r"out\.electricity\..*\.energy_consumption\.\.kwh")]
    .loc[:, lambda x: x.ne(0).any(axis=0)]
    .rename(columns=lambda x: x[16:-24])
    .drop(columns="net")
)

dfh = dff.resample("h").sum()
dfd = dfh.resample("D").sum()

fig, ax = plt.subplots(figsize=(12, 8), constrained_layout=True)
dfd.drop(columns="total").plot(ax=ax, kind="area", cmap="tab20")
plt.legend(loc="upper right")
plt.show()

In [ ]:
piv = dfh.pivot_table(index=dfh.index.normalize(), columns=dfh.index.hour, values="total")
cmap = (piv.index.day_of_week // 5).map({0: "tab:blue", 1: "tab:red"})
fig, ax = plt.subplots(figsize=(12, 8), constrained_layout=True)
piv.T.plot(ax=ax, color=cmap, alpha=0.1, legend=False)
plt.show()

## Sélection de 100 bâtiments tout-électriques

On réutilise le filtre déjà validé dans `extraction_timeseries_oedi.ipynb` (maison individuelle 1 étage, chauffage électrique gainé, Central AC, sans VE/piscine/PV, zones climatiques 3A/4A/5A), auquel on ajoute un critère **strict tout-électrique** : consommation nulle de gaz naturel, fioul et propane (`out.*.total.energy_consumption..kwh == 0`). Cela garantit que les bâtiments étudiés n'ont aucun usage énergétique caché sur un autre vecteur (chauffage secondaire au gaz, chauffe-eau au fioul, etc.), condition nécessaire pour analyser proprement la flexibilité de la demande électrique.

In [ ]:
N_BUILDINGS = 100
BUILDINGS_LIST_PATH = DATA_PROCESSED / "buildings_all_electric_100.parquet"

raw_meta = pd.read_parquet(DATA_RAW / "upgrade0.parquet", columns=[
    "bldg_id", "in.state", "in.geometry_building_type_recs", "in.geometry_stories",
    "in.electric_vehicle_ownership", "in.misc_pool", "in.has_pv",
    "in.hvac_cooling_type", "in.hvac_heating_type", "in.heating_fuel",
    "in.ashrae_iecc_climate_zone_2004",
    "out.natural_gas.total.energy_consumption..kwh",
    "out.fuel_oil.total.energy_consumption..kwh",
    "out.propane.total.energy_consumption..kwh",
])

fuel_totals = [
    "out.natural_gas.total.energy_consumption..kwh",
    "out.fuel_oil.total.energy_consumption..kwh",
    "out.propane.total.energy_consumption..kwh",
]
all_electric_mask = (raw_meta[fuel_totals].fillna(0) == 0).all(axis=1)

filter_mask = (
    (raw_meta["in.geometry_building_type_recs"] == "Single-Family Detached") &
    (raw_meta["in.geometry_stories"] == "1") &
    (raw_meta["in.electric_vehicle_ownership"] == "No") &
    (raw_meta["in.misc_pool"] == "None") &
    (raw_meta["in.has_pv"] == "No") &
    (raw_meta["in.hvac_cooling_type"] == "Central AC") &
    (raw_meta["in.hvac_heating_type"] == "Ducted Heating") &
    (raw_meta["in.heating_fuel"] == "Electricity") &
    (raw_meta["in.ashrae_iecc_climate_zone_2004"].isin(["3A", "4A", "5A"])) &
    all_electric_mask
)

candidates = raw_meta.loc[filter_mask, ["bldg_id", "in.state", "in.ashrae_iecc_climate_zone_2004"]].reset_index(drop=True)
print(f"{len(candidates):,} bâtiments candidats (filtre existant + tout-électrique strict)")

if BUILDINGS_LIST_PATH.exists():
    buildings_100 = pd.read_parquet(BUILDINGS_LIST_PATH)
else:
    buildings_100 = candidates.sample(n=N_BUILDINGS, random_state=42).reset_index(drop=True)
    buildings_100.to_parquet(BUILDINGS_LIST_PATH, index=False)

print(f"Bâtiments retenus : {len(buildings_100)}")
buildings_100.head()

## Téléchargement des séries temporelles manquantes

Les séries des 100 bâtiments sont stockées dans `data/raw/timeseries_100_all_electric/`, séparément des quelques bâtiments d'exemple présents directement dans `data/raw/`. La cellule ci-dessous est idempotente : elle ne télécharge que les fichiers manquants.

In [ ]:
OEDI_BASE = (
    "https://oedi-data-lake.s3.amazonaws.com/"
    "nrel-pds-building-stock/end-use-load-profiles-for-us-building-stock/"
    "2025/resstock_amy2018_release_1/"
    "timeseries_individual_buildings/by_state/upgrade=0"
)

DATA_RAW_100.mkdir(parents=True, exist_ok=True)

ok, failed = 0, []
for _, row in buildings_100.iterrows():
    bid, state = int(row["bldg_id"]), row["in.state"]
    out = DATA_RAW_100 / f"{bid}-0.parquet"
    if out.exists():
        ok += 1
        continue
    url = f"{OEDI_BASE}/state={state}/{bid}-0.parquet"
    try:
        df_ts = pd.read_parquet(url)
        df_ts.to_parquet(out)
        ok += 1
    except Exception as e:
        failed.append((bid, state, str(e)))

print(f"Séries disponibles : {ok} / {len(buildings_100)}")
if failed:
    print(f"Échecs ({len(failed)}) :", failed)

## Fonction d'extraction jour × features (un bâtiment)

In [ ]:
def build_day_features(parquet_path, col=COL, weather_cols=WEATHER_COLS, freq=RESAMPLE_FREQ):
    """Extrait les jours (forme + amplitude + météo) pour un bâtiment, au pas horaire."""
    d = pd.read_parquet(parquet_path)
    d = d.assign(timestamp=lambda x: pd.to_datetime(x["timestamp"]) - pd.Timedelta("15m")).set_index("timestamp")

    agg = {col: "sum"}
    for wcol in weather_cols:
        if wcol in d.columns:
            agg[wcol] = "mean"
    d_h = d[list(agg)].resample(freq).agg(agg)

    ts = d_h[col].values
    n_days_ = len(ts) // steps_per_day
    ts = ts[: n_days_ * steps_per_day]

    dates_ = d_h.index[: n_days_ * steps_per_day]

    days_ = ts.reshape(n_days_, steps_per_day)
    dates_daily = (
        pd.Series(dates_)
        .groupby(np.arange(len(dates_)) // steps_per_day)
        .first().dt.normalize().values
    )

    mask_valid_ = ~np.isnan(days_).any(axis=1)
    days_ = days_[mask_valid_]
    dates_daily = dates_daily[mask_valid_]

    # Forme
    mean_ = days_.mean(axis=1, keepdims=True)
    std_ = days_.std(axis=1, keepdims=True)
    std_[std_ == 0] = 1e-8
    shape_ = (days_ - mean_) / std_

    # Amplitude
    amplitude_ = days_.sum(axis=1)

    meta = {
        "bldg_id": Path(parquet_path).stem,
        "date": dates_daily,
        "amplitude": amplitude_,
    }

    for wcol in weather_cols:
        prefix = wcol.replace("out.", "").replace("..", "_").replace(".", "_")
        if wcol in d_h.columns:
            weather = d_h[wcol].values[: n_days_ * steps_per_day].reshape(n_days_, steps_per_day)[mask_valid_]
            meta[f"{prefix}_mean"] = weather.mean(axis=1)
            meta[f"{prefix}_min"] = weather.min(axis=1)
            meta[f"{prefix}_max"] = weather.max(axis=1)
        else:
            meta[f"{prefix}_mean"] = np.nan
            meta[f"{prefix}_min"] = np.nan
            meta[f"{prefix}_max"] = np.nan

    return pd.DataFrame(meta), shape_, days_, dates_daily

## Extraction multi-bâtiments (100 bâtiments tout-électriques, pas horaire)

In [ ]:
parquet_files = sorted(DATA_RAW_100.glob("*-0.parquet"))
print(f"Fichiers disponibles : {len(parquet_files)} / {len(buildings_100)}")

all_meta, all_shapes, all_days_raw = [], [], []

for f in parquet_files:
    meta_f, shape_f, days_raw_f, _ = build_day_features(f)
    all_meta.append(meta_f)
    all_shapes.append(shape_f)
    all_days_raw.append(days_raw_f)

meta_all = pd.concat(all_meta, ignore_index=True)
shape_all = np.vstack(all_shapes)
days_raw_all = np.vstack(all_days_raw)

print("meta :", meta_all.shape)
print("shape:", shape_all.shape)
print("Nombre de bâtiments :", meta_all["bldg_id"].nunique())

## Clustering exploratoire mono-bâtiment (forme vs amplitude) sur le bâtiment de référence

In [ ]:
bldg_ref = sorted(meta_all["bldg_id"].unique())[0]
mask_ref = meta_all["bldg_id"].values == bldg_ref
print(f"Bâtiment de référence : {bldg_ref}")

days_clean = days_raw_all[mask_ref]
day_dates_clean = pd.to_datetime(meta_all.loc[mask_ref, "date"].values)
shape_ref = shape_all[mask_ref]

day_mean = days_clean.mean(axis=1, keepdims=True)
day_std = days_clean.std(axis=1, keepdims=True)
day_std[day_std == 0] = 1e-8

X = (days_clean - day_mean) / day_std          # forme
Y = StandardScaler().fit_transform(days_clean)  # amplitude/niveau global

print(X.shape, Y.shape)

## Fonctions utilitaires (coude/silhouette, PCA, projection)

In [ ]:
def plot_cluster_profiles(X, title1, title2):
    k_range = range(2, 15)
    inertias, silhouettes = [], []

    for k in k_range:
        km = KMeans(n_clusters=k, random_state=42, n_init=10)
        labels = km.fit_predict(X)
        inertias.append(km.inertia_)
        sil = silhouette_score(X, labels, sample_size=5000, random_state=42)
        silhouettes.append(sil)
        print(f"k={k:2d} | inertia={km.inertia_:10.1f} | silhouette={sil:.4f}")

    k_opt = KneeLocator(list(k_range), inertias, curve="convex", direction="decreasing").elbow
    print(f"K optimal (coude) : {k_opt}")

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].plot(list(k_range), inertias, marker='o')
    axes[0].set_xlabel("k"); axes[0].set_ylabel("Inertie"); axes[0].set_title(title1)
    axes[1].plot(list(k_range), silhouettes, marker='o', color='orange')
    axes[1].set_xlabel("k"); axes[1].set_ylabel("Silhouette"); axes[1].set_title(title2)
    plt.tight_layout()
    plt.show()
    return k_opt


def cluster_days(X_input, day_dates_input, k):
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=20)
    cluster_labels = kmeans.fit_predict(X_input)

    results = pd.DataFrame({"date": day_dates_input, "cluster": cluster_labels})
    results["weekday"] = results["date"].dt.day_name()
    results["is_weekend"] = results["date"].dt.dayofweek >= 5
    results["month"] = results["date"].dt.month

    print(results["cluster"].value_counts().sort_index())
    return kmeans, cluster_labels, results


def compute_pca(X_input, n_components=10, titre1=""):
    pca = PCA(n_components=n_components, random_state=42)
    X_pca = pca.fit_transform(X_input)
    var_ratio = pca.explained_variance_ratio_
    var_cum = np.cumsum(var_ratio)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].bar(range(1, len(var_ratio) + 1), var_ratio)
    axes[0].set_xlabel("Composante"); axes[0].set_ylabel("Variance expliquée")
    axes[0].set_title("Variance expliquée par composante")
    axes[1].plot(range(1, len(var_cum) + 1), var_cum, marker="o")
    axes[1].axhline(0.90, color="red", linestyle="--", label="90 %")
    axes[1].set_xlabel("Nb composantes"); axes[1].set_ylabel("Variance cumulée")
    axes[1].legend(); axes[1].set_title(f"Variance cumulée {titre1}")
    plt.tight_layout()
    plt.show()

    n_components_90 = np.argmax(var_cum >= 0.90) + 1
    print(f"Composantes pour 90% variance {titre1}: {n_components_90}")
    return X_pca, pca, var_ratio, var_cum, n_components_90


def plot_pca_clusters(X_pca, cluster_labels, var_ratio, title="Projection PCA"):
    fig, ax = plt.subplots(figsize=(8, 7))
    scatter = ax.scatter(X_pca[:, 0], X_pca[:, 1], c=cluster_labels, cmap="tab10", s=8, alpha=0.6)
    ax.set_xlabel(f"PC1 ({var_ratio[0]*100:.1f}%)")
    ax.set_ylabel(f"PC2 ({var_ratio[1]*100:.1f}%)")
    ax.set_title(title)
    plt.colorbar(scatter, ax=ax, label="Cluster")
    plt.tight_layout()
    plt.show()


def plot_cluster_scatter_2d(x, y, cluster_labels, xlabel, ylabel, title):
    """Scatter 2D interprétable (pas de PCA) pour visualiser les clusters selon deux variables explicites."""
    fig, ax = plt.subplots(figsize=(8, 7))
    scatter = ax.scatter(x, y, c=cluster_labels, cmap="tab10", s=8, alpha=0.6)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    plt.colorbar(scatter, ax=ax, label="Cluster")
    plt.tight_layout()
    plt.show()


def plot_interactive_all_curves_by_cluster(days_matrix, cluster_labels, k_final, hours, titre, y_label="Consommation (kWh)"):
    """Visualisation interactive de TOUTES les courbes journalières (pas seulement la moyenne),
    superposées par cluster (technique 'spaghetti plot' : une seule trace par cluster,
    segments séparés par des NaN pour rester performant même avec des dizaines de milliers de courbes).
    Utilise go.Scatter (SVG) plutôt que Scattergl : évite toute dépendance à WebGL, qui n'est pas
    toujours disponible dans le webview de VS Code."""
    fig = go.Figure()

    for c in range(k_final):
        mask = cluster_labels == c
        profiles = days_matrix[mask]
        n_curves = profiles.shape[0]

        x_spaghetti = np.tile(np.append(hours, np.nan), n_curves)
        y_spaghetti = np.concatenate([np.append(p, np.nan) for p in profiles])

        fig.add_trace(go.Scatter(
            x=x_spaghetti, y=y_spaghetti, mode="lines",
            line=dict(width=1, color="rgba(31,119,180,0.08)"),
            name=f"Cluster {c} — {n_curves} jours", visible=(c == 0),
            hoverinfo="skip", showlegend=False,
        ))

        mean_profile = profiles.mean(axis=0)
        fig.add_trace(go.Scatter(
            x=hours, y=mean_profile, mode="lines",
            line=dict(width=3, color="black"),
            name=f"Moyenne cluster {c} (n={n_curves})", visible=(c == 0),
        ))

    buttons = []
    for c in range(k_final):
        visibility = []
        for i in range(k_final):
            visibility += [i == c, i == c]
        buttons.append(dict(label=f"Cluster {c}", method="update",
                             args=[{"visible": visibility}, {"title": f"{titre} — Cluster {c}"}]))
    buttons.append(dict(label="Tous", method="update",
                         args=[{"visible": [True] * (2 * k_final)}, {"title": f"{titre} — tous les clusters"}]))

    fig.update_layout(
        title=f"{titre} — Cluster 0", xaxis_title="Heure", yaxis_title=y_label,
        template="plotly_white", width=1050, height=650,
        updatemenus=[dict(buttons=buttons, direction="down", x=1.05, y=1, showactive=True)]
    )
    fig.show()


def plot_interactive_building_days(bldg_id, meta_df, days_matrix, cluster_labels, hours, k_final):
    """Toutes les journées d'UN bâtiment, colorées par cluster (utile pour l'inspection fine)."""
    mask = meta_df["bldg_id"].values == bldg_id
    profiles = days_matrix[mask]
    clusters = np.asarray(cluster_labels)[mask]

    fig = go.Figure()
    colors = plt.cm.tab10(np.linspace(0, 1, k_final))
    for c in range(k_final):
        cmask = clusters == c
        if cmask.sum() == 0:
            continue
        profs = profiles[cmask]
        x_sp = np.tile(np.append(hours, np.nan), profs.shape[0])
        y_sp = np.concatenate([np.append(p, np.nan) for p in profs])
        r, g, b = (int(255 * v) for v in colors[c][:3])
        fig.add_trace(go.Scatter(
            x=x_sp, y=y_sp, mode="lines",
            line=dict(width=1.5, color=f"rgba({r},{g},{b},0.5)"),
            name=f"Cluster {c} (n={cmask.sum()})",
        ))
    fig.update_layout(
        title=f"Bâtiment {bldg_id} — toutes les journées colorées par cluster",
        xaxis_title="Heure", yaxis_title="Consommation (kWh)",
        template="plotly_white", width=1000, height=600,
    )
    fig.show()

## Choix de k et clustering (forme vs amplitude, bâtiment de référence)

In [ ]:
k_forme  = plot_cluster_profiles(X, "Coude - forme", "Silhouette - forme")
k_amplet = plot_cluster_profiles(Y, "Coude - amplitude", "Silhouette - amplitude")

kmeans_forme, cluster_labels_forme, results_forme = cluster_days(X, day_dates_clean, k_forme)
kmeans_amplet, cluster_labels_amplet, results_amplet = cluster_days(Y, day_dates_clean, k_amplet)

## Visualisation interactive : toutes les courbes journalières (pas seulement la moyenne)

In [ ]:
plot_interactive_all_curves_by_cluster(days_clean, cluster_labels_amplet, k_amplet, HOURS, "Profil (toutes les journées) — amplitude")
plot_interactive_all_curves_by_cluster(days_clean, cluster_labels_forme, k_forme, HOURS, "Profil (toutes les journées) — forme")

## Décomposition par poste de consommation (bâtiment de référence)

Comme dans `timeseries_clustering.ipynb` : au lieu de la seule courbe totale par cluster, on
décompose chaque cluster (forme / amplitude) du bâtiment de référence par poste de consommation
électrique (`out.electricity.<poste>.energy_consumption..kwh`, hors `total`/`net`).

In [ ]:
ENDUSE_PATTERN = r"out\.electricity\..*\.energy_consumption\.\.kwh"


def extract_enduse_days_hourly(parquet_path, dates_ref, freq=RESAMPLE_FREQ, steps_per_day=steps_per_day):
    """Même découpage jour × poste que build_day_features (résample au pas horaire, reshape
    en jours), mais pour chaque poste de consommation électrique (hors total/net) au lieu de
    la seule colonne COL. Restreint aux jours de `dates_ref` (ceux retenus par le pipeline
    principal pour ce bâtiment, après suppression des jours invalides)."""
    d = pd.read_parquet(parquet_path)
    d = d.assign(timestamp=lambda x: pd.to_datetime(x["timestamp"]) - pd.Timedelta("15m")).set_index("timestamp")

    enduse_mask = d.columns.str.match(ENDUSE_PATTERN)
    enduse_cols = [c for c in d.columns[enduse_mask] if c[16:-24] not in ("total", "net")]
    d_h = d[enduse_cols].resample(freq).sum()

    n_days_ = len(d_h) // steps_per_day
    d_h = d_h.iloc[: n_days_ * steps_per_day]

    dates_daily = (
        pd.Series(d_h.index)
        .groupby(np.arange(len(d_h)) // steps_per_day)
        .first().dt.normalize().values
    )
    keep = np.isin(dates_daily, dates_ref)

    enduse_days = {}
    for col in enduse_cols:
        poste = col[16:-24]
        vals = d_h[col].values.reshape(n_days_, steps_per_day)[keep]
        enduse_days[poste] = vals

    return enduse_days


def plot_cluster_profiles_by_enduse(enduse_days, cluster_labels, k_final, titre, hours=HOURS):
    """Pour chaque cluster, courbe moyenne empilée PAR POSTE de consommation
    (chauffage, eau chaude, éclairage, électroménager...) au lieu de la seule courbe totale."""
    postes = list(enduse_days.keys())
    n_cols = min(3, k_final)
    n_rows = int(np.ceil(k_final / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(6 * n_cols, 4 * n_rows), sharex=True, sharey=True)
    axes = np.atleast_1d(axes).flatten()

    for c in range(k_final):
        mask = cluster_labels == c
        mean_by_poste = {poste: np.nanmean(enduse_days[poste][mask], axis=0) for poste in postes}
        stacked = pd.DataFrame(mean_by_poste, index=hours)
        stacked.plot(kind="area", ax=axes[c], cmap="tab20", legend=(c == 0), linewidth=0)
        axes[c].set_title(f"Cluster {c} (n={mask.sum()})")
        axes[c].set_xlabel("Heure")
        axes[c].set_ylabel("kWh")

    for ax in axes[k_final:]:
        ax.set_visible(False)

    fig.suptitle(titre, fontsize=14)
    plt.tight_layout()
    plt.show()


enduse_days_ref = extract_enduse_days_hourly(
    DATA_RAW_100 / f"{bldg_ref}.parquet", dates_ref=day_dates_clean
)
print(f"Postes de consommation détectés ({bldg_ref}) :", list(enduse_days_ref.keys()))

plot_cluster_profiles_by_enduse(
    enduse_days_ref, cluster_labels_forme, k_forme,
    f"{bldg_ref} — Décomposition par poste — profils de forme"
)
plot_cluster_profiles_by_enduse(
    enduse_days_ref, cluster_labels_amplet, k_amplet,
    f"{bldg_ref} — Décomposition par poste — profils d'amplitude"
)

## Composition calendaire des clusters

In [ ]:
def plot_cluster_calendar_composition(results, titre):
    comp_weekend = pd.crosstab(results["cluster"], results["is_weekend"], normalize="index") * 100
    comp_weekend.columns = ["Semaine (%)", "Weekend (%)"]
    comp_month = pd.crosstab(results["cluster"], results["month"], normalize="index") * 100

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    comp_weekend.plot(kind="bar", stacked=True, ax=axes[0], colormap="Set2")
    axes[0].set_title("Semaine / weekend par cluster")
    sns.heatmap(comp_month, annot=True, fmt=".0f", cmap="YlOrRd", ax=axes[1])
    axes[1].set_title("Répartition mensuelle par cluster (%)")
    fig.suptitle(titre, fontsize=14)
    plt.tight_layout()
    plt.show()
    return comp_weekend, comp_month

plot_cluster_calendar_composition(results_amplet, "Composition (amplitude)")
plot_cluster_calendar_composition(results_forme, "Composition (forme)")

## Frise temporelle

In [ ]:
def plot_cluster_timeline(results, k_final, titre):
    results_sorted = results.sort_values("date").reset_index(drop=True)
    fig, ax = plt.subplots(figsize=(16, 3))
    scatter = ax.scatter(results_sorted["date"], [1] * len(results_sorted),
                          c=results_sorted["cluster"], cmap="tab10", s=15)
    ax.set_yticks([])
    ax.set_title(titre)
    plt.colorbar(scatter, ax=ax, label="Cluster", ticks=range(k_final))
    plt.tight_layout()
    plt.show()

plot_cluster_timeline(results_forme, k_forme, "Timeline clusters (forme)")
plot_cluster_timeline(results_amplet, k_amplet, "Timeline clusters (amplitude)")

## PCA sur forme et amplitude (bâtiment de référence)

In [ ]:
X_pca_forme, pca_forme, var_ratio_forme, var_cum_forme, _ = compute_pca(X, n_components=10, titre1="(Forme)")
Y_pca_amplet, pca_amplet, var_ratio_amplet, var_cum_amplet, _ = compute_pca(Y, n_components=10, titre1="(Amplet)")

plot_pca_clusters(X_pca_forme, cluster_labels_forme, var_ratio_forme, "PCA clusters (forme)")
plot_pca_clusters(Y_pca_amplet, cluster_labels_amplet, var_ratio_amplet, "PCA clusters (amplitude)")

## Dataset multivarié multi-bâtiments (forme + amplitude z-scorée/bâtiment + météo globale)

In [ ]:
weather_feature_cols = [c for c in meta_all.columns if c.endswith(("_mean", "_min", "_max"))]
print("Colonnes météo :", weather_feature_cols)

mask_ok = meta_all[weather_feature_cols].notna().all(axis=1).values
print(f"Jours conservés : {mask_ok.sum()} / {len(mask_ok)}")

shape_ok = shape_all[mask_ok]
days_raw_ok = days_raw_all[mask_ok]
meta_ok = meta_all[mask_ok].reset_index(drop=True)

print("Nombre de bâtiments :", meta_ok["bldg_id"].nunique())

In [ ]:
# Amplitude : log + z-score PAR BÂTIMENT (isole le jour type de la taille du bâtiment)
meta_ok["amp_log"] = np.log1p(meta_ok["amplitude"])

amp_stats = meta_ok.groupby("bldg_id")["amp_log"].agg(amp_log_mean="mean", amp_log_std="std")
amp_stats["amp_log_std"] = amp_stats["amp_log_std"].replace(0, np.nan).fillna(1e-8)

meta_ok = meta_ok.merge(amp_stats, on="bldg_id", how="left")
meta_ok["amplitude_z"] = (meta_ok["amp_log"] - meta_ok["amp_log_mean"]) / meta_ok["amp_log_std"]

# Météo : standardisation globale — gardée TELLE QUELLE (pas de PCA), pour rester interprétable
# dans les analyses de corrélation qui suivent
weather_scaler = StandardScaler()
weather_scaled = weather_scaler.fit_transform(meta_ok[weather_feature_cols].values)

scalar_features = np.column_stack([meta_ok["amplitude_z"].values, weather_scaled])
print("Forme (à réduire par PCA) :", shape_ok.shape, "| Amplitude + météo (brutes) :", scalar_features.shape)

## PCA (sur la forme uniquement) + choix de k (multivarié multi-bâtiments)

La PCA est appliquée uniquement sur les 24 points de forme journalière (`shape_ok`), pour réduire leur
dimensionnalité tout en gardant l'essentiel de l'information de courbe. Les variables d'amplitude et de
météo sont standardisées mais **ne passent pas par la PCA** : elles restent directement interprétables
dans l'analyse de corrélation qui suit.

In [ ]:
shape_pca, pca_shape, var_ratio_shape, var_cum_shape, _ = compute_pca(
    shape_ok, n_components=10, titre1="(Forme, multi-bâtiments)"
)

# Concaténation : forme réduite par PCA + amplitude/météo brutes standardisées (PAS de PCA sur la météo)
X_multi = np.hstack([shape_pca, scalar_features]).astype(np.float64)
print("X_multi :", X_multi.shape)

k_multi_opt = plot_cluster_profiles(
    X_multi,
    "Coude - Clustering multivarié multi-bâtiments",
    "Silhouette - Clustering multivarié multi-bâtiments"
)

## Clustering final multivarié

In [ ]:
kmeans_multi = KMeans(n_clusters=k_multi_opt, random_state=42, n_init=20)
cluster_labels_multi = kmeans_multi.fit_predict(X_multi)

meta_ok["cluster_multi"] = cluster_labels_multi
meta_ok["weekday"] = pd.to_datetime(meta_ok["date"]).dt.day_name()
meta_ok["is_weekend"] = pd.to_datetime(meta_ok["date"]).dt.dayofweek >= 5
meta_ok["month"] = pd.to_datetime(meta_ok["date"]).dt.month

print(meta_ok["cluster_multi"].value_counts().sort_index())

# Scatter interprétable : PC1 de la forme (axe X) vs température extérieure moyenne (axe Y)
# — pas de PCA sur la météo, on l'affiche directement
main_weather_col = next((c for c in weather_feature_cols if c.endswith("_mean")), weather_feature_cols[0])
plot_cluster_scatter_2d(
    shape_pca[:, 0], meta_ok[main_weather_col].values, cluster_labels_multi,
    xlabel=f"PC1 de la forme ({var_ratio_shape[0]*100:.1f}%)", ylabel=main_weather_col,
    title=f"Clustering multivarié multi-bâtiments, k={k_multi_opt}"
)

## Vérification : un cluster n'est pas dominé par un seul bâtiment

In [ ]:
comp_bldg = pd.crosstab(meta_ok["cluster_multi"], meta_ok["bldg_id"], normalize="index") * 100

fig, ax = plt.subplots(figsize=(12, 5))
sns.heatmap(comp_bldg, cmap="YlGnBu", ax=ax)
ax.set_title("Répartition (%) des bâtiments par cluster")
plt.tight_layout()
plt.show()

## Analyse des clusters multivariés

In [ ]:
main_weather_col = next((c for c in weather_feature_cols if c.endswith("_mean")), weather_feature_cols[0])

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.boxplot(data=meta_ok, x="cluster_multi", y=main_weather_col, ax=axes[0])
axes[0].set_title(f"{main_weather_col} par cluster")
sns.boxplot(data=meta_ok, x="cluster_multi", y="amplitude_z", ax=axes[1])
axes[1].set_title("Amplitude relative (z-score/bâtiment) par cluster")
comp_weekend_multi = pd.crosstab(meta_ok["cluster_multi"], meta_ok["is_weekend"], normalize="index") * 100
comp_weekend_multi.columns = ["Semaine (%)", "Weekend (%)"]
comp_weekend_multi.plot(kind="bar", stacked=True, ax=axes[2], colormap="Set2")
axes[2].set_title("Semaine / weekend par cluster")
plt.tight_layout()
plt.show()

In [ ]:
mean_cols = [c for c in weather_feature_cols if c.endswith("_mean")] + ["amplitude_z"]
cluster_summary = meta_ok.groupby("cluster_multi")[mean_cols].mean()

fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(cluster_summary.T, annot=True, fmt=".1f", cmap="coolwarm", ax=ax)
ax.set_title("Moyenne des variables par cluster")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))
colors = plt.cm.tab10(np.linspace(0, 1, k_multi_opt))
for c in range(k_multi_opt):
    mask = cluster_labels_multi == c
    ax.plot(HOURS, shape_ok[mask].mean(axis=0), label=f"Cluster {c} (n={mask.sum()})", color=colors[c], linewidth=2)
ax.set_xlabel("Heure"); ax.set_ylabel("Consommation normalisée (forme)")
ax.set_title("Profils de forme moyens par cluster (multi-bâtiments)")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Visualisation interactive : toutes les courbes de consommation (100 bâtiments)

Version interactive du graphique précédent : au lieu de n'afficher que la moyenne, on superpose
**toutes** les journées de consommation réelle (kWh, pas de forme normalisée) de chaque cluster, avec
la moyenne en surimpression. Les boutons permettent de basculer entre clusters ou de tous les afficher.

In [ ]:
plot_interactive_all_curves_by_cluster(
    days_raw_ok, cluster_labels_multi, k_multi_opt, HOURS,
    "Consommation réelle — toutes les journées par cluster (100 bâtiments)"
)

# Inspection d'un bâtiment particulier : toutes ses journées, colorées par cluster
example_bldg = meta_ok["bldg_id"].iloc[0]
plot_interactive_building_days(example_bldg, meta_ok, days_raw_ok, cluster_labels_multi, HOURS, k_multi_opt)

## Décomposition par poste de consommation (clustering multivarié multi-bâtiments)

Même principe que pour le bâtiment de référence, mais appliqué au clustering multivarié pooling
les 100 bâtiments (`cluster_labels_multi`) : pour chaque cluster, la courbe moyenne empilée par
poste plutôt que la seule courbe totale. Comme `build_day_features` (cellule plus haut) ne
conserve que le total et la météo par bâtiment, on relit ici les fichiers bruts pour extraire les
postes de consommation, en les alignant sur les jours déjà retenus dans `meta_ok`.

*(Coût : relit les 100 fichiers parquet une seconde fois — attendu, potentiellement plusieurs
dizaines de secondes.)*

In [ ]:
def extract_enduse_days_multi(parquet_files, meta_ok, freq=RESAMPLE_FREQ, steps_per_day=steps_per_day):
    """Pour chaque bâtiment, extrait les postes de consommation (hors total/net) au même pas
    horaire que build_day_features, puis ne garde que les jours déjà présents dans `meta_ok`
    (filtrés par le pipeline principal — météo complète), alignés sur son ordre (bldg_id, date).
    Retourne un dict {poste: array(len(meta_ok), steps_per_day)} (NaN si le bâtiment n'a pas
    ce poste, ou si le jour n'a pas été retrouvé)."""
    n_rows = len(meta_ok)
    enduse_days = {}

    for f in parquet_files:
        bldg_id = f.stem
        rows_mask = (meta_ok["bldg_id"].values == bldg_id)
        if not rows_mask.any():
            continue  # bâtiment filtré par mask_ok (météo incomplète)

        d = pd.read_parquet(f)
        d = d.assign(timestamp=lambda x: pd.to_datetime(x["timestamp"]) - pd.Timedelta("15m")).set_index("timestamp")

        enduse_mask = d.columns.str.match(ENDUSE_PATTERN)
        enduse_cols = [c for c in d.columns[enduse_mask] if c[16:-24] not in ("total", "net")]
        if not enduse_cols:
            continue
        d_h = d[enduse_cols].resample(freq).sum()

        n_days_ = len(d_h) // steps_per_day
        d_h = d_h.iloc[: n_days_ * steps_per_day]
        dates_daily = (
            pd.Series(d_h.index)
            .groupby(np.arange(len(d_h)) // steps_per_day)
            .first().dt.normalize().values
        )
        date_to_pos = {pd.Timestamp(dt): i for i, dt in enumerate(dates_daily)}

        target_rows = meta_ok.index[rows_mask]
        target_dates = meta_ok.loc[rows_mask, "date"].values

        for col in enduse_cols:
            poste = col[16:-24]
            if poste not in enduse_days:
                enduse_days[poste] = np.full((n_rows, steps_per_day), np.nan)
            vals = d_h[col].values.reshape(n_days_, steps_per_day)
            for row, date in zip(target_rows, target_dates):
                pos = date_to_pos.get(pd.Timestamp(date))
                if pos is not None:
                    enduse_days[poste][row] = vals[pos]

    return enduse_days


enduse_days_multi = extract_enduse_days_multi(parquet_files, meta_ok)
print("Postes de consommation détectés (multi-bâtiments) :", list(enduse_days_multi.keys()))

plot_cluster_profiles_by_enduse(
    enduse_days_multi, cluster_labels_multi, k_multi_opt,
    "Décomposition par poste — clustering multivarié multi-bâtiments"
)

## Corrélations : quelles variables expliquent la variation de consommation ?

On corrèle l'amplitude journalière (niveau de consommation, indépendant de la taille du bâtiment grâce
au z-score), les variables météo, le calendrier (weekend, mois) et la forme (PC1) entre eux. Cela aide à
distinguer ce qui pilote la variation de consommation : température extérieure, humidité, saisonnalité,
ou comportement (semaine/weekend).

In [ ]:
corr_df = meta_ok[weather_feature_cols + ["amplitude_z"]].copy()
corr_df["is_weekend"] = meta_ok["is_weekend"].astype(int)
corr_df["month"] = meta_ok["month"]
corr_df["shape_pc1"] = shape_pca[:, 0]
corr_df["shape_pc2"] = shape_pca[:, 1]

corr_matrix = corr_df.corr()

fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
ax.set_title("Matrice de corrélation — météo, calendrier, forme, amplitude")
plt.tight_layout()
plt.show()

In [ ]:
# Corrélation de chaque variable avec l'amplitude (niveau de consommation), triée par force d'association
corr_with_amp = corr_matrix["amplitude_z"].drop("amplitude_z").sort_values(key=np.abs, ascending=False)

fig, ax = plt.subplots(figsize=(9, 6))
colors_corr = ["#e74c3c" if v < 0 else "#2ecc71" for v in corr_with_amp]
ax.barh(corr_with_amp.index, corr_with_amp.values, color=colors_corr, alpha=0.85)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Corrélation avec l'amplitude (z-score)")
ax.set_title("Variables les plus associées à la variation de consommation")
ax.invert_yaxis()
plt.tight_layout()
plt.show()
print(corr_with_amp)

In [ ]:
# Scatter interactif : variable météo la plus corrélée vs amplitude, coloré par cluster, avec droite de tendance
top_weather_var = corr_with_amp.drop(labels=["is_weekend", "month", "shape_pc1", "shape_pc2"], errors="ignore").index[0]

fig_corr = px.scatter(
    meta_ok.assign(shape_pc1=shape_pca[:, 0], cluster=cluster_labels_multi.astype(str)),
    x=top_weather_var, y="amplitude_z", color="cluster",
    trendline="ols", opacity=0.35, render_mode="svg",  # évite le bascule auto en WebGL (Scattergl) au-delà de ~1000 points
    title=f"Amplitude vs {top_weather_var} (corrélation = {corr_with_amp[top_weather_var]:.2f})",
    labels={"amplitude_z": "Amplitude (z-score/bâtiment)", top_weather_var: top_weather_var},
    template="plotly_white", width=900, height=600,
)
fig_corr.show()

## Sauvegarde de tous les artefacts

In [ ]:
"""meta_ok.to_parquet(DATA_PROCESSED / "clustering_multivarie_jours_types.parquet", index=False)

joblib.dump(kmeans_multi, DATA_PROCESSED / "kmeans_multivarie.joblib")
joblib.dump(pca_shape, DATA_PROCESSED / "pca_shape.joblib")
joblib.dump(weather_scaler, DATA_PROCESSED / "weather_scaler.joblib")
joblib.dump(amp_stats, DATA_PROCESSED / "amp_stats.joblib")
joblib.dump(weather_feature_cols, DATA_PROCESSED / "weather_feature_cols.joblib")
"""

## Reverse : classifier un nouveau jour dans un cluster existant

In [ ]:
def classify_new_day(
    profile_kwh,
    weather_values,
    bldg_id=None,
    amp_stats=amp_stats,
    weather_scaler=weather_scaler,
    weather_feature_cols=weather_feature_cols,
    pca_shape=pca_shape,
    kmeans=kmeans_multi,
):
    """
    profile_kwh : array (steps_per_day,) — profil horaire brut du jour à classer
    weather_values : dict {nom_colonne_meteo: valeur}
    bldg_id : identifiant du bâtiment
    """
    profile_kwh = np.asarray(profile_kwh, dtype=np.float64)
    assert profile_kwh.shape[0] == steps_per_day, \
        f"Le profil doit avoir {steps_per_day} points"

    # Forme
    m, s = profile_kwh.mean(), profile_kwh.std()
    if s == 0:
        s = 1e-8
    shape_new = (profile_kwh - m) / s

    # Amplitude
    amp_log_new = np.log1p(profile_kwh.sum())
    if bldg_id is not None and bldg_id in amp_stats.index:
        mu = amp_stats.loc[bldg_id, "amp_log_mean"]
        sd = amp_stats.loc[bldg_id, "amp_log_std"]
    else:
        mu = amp_stats["amp_log_mean"].mean()
        sd = amp_stats["amp_log_std"].mean()
    if sd == 0:
        sd = 1e-8
    amp_z = (amp_log_new - mu) / sd

    # Météo (standardisée, pas de PCA)
    weather_vec = np.array([[weather_values[c] for c in weather_feature_cols]], dtype=np.float64)
    weather_scaled_new = weather_scaler.transform(weather_vec)[0]

    # Forme réduite par la PCA ajustée sur l'entraînement
    shape_pca_new = pca_shape.transform(shape_new.reshape(1, -1))[0]

    # Assemblage : même ordre que X_multi = [shape_pca, amplitude_z, weather_scaled]
    full_vec = np.concatenate([shape_pca_new, [amp_z], weather_scaled_new]).reshape(1, -1).astype(np.float64)

    return int(kmeans.predict(full_vec)[0])

## Test de cohérence (reclasser un jour déjà connu)

In [ ]:
idx_test = 0
row_test = meta_ok.iloc[idx_test]
bldg_test = row_test["bldg_id"]
weather_test = {c: row_test[c] for c in weather_feature_cols}

# Récupération du profil brut correspondant (même ligne dans days_raw_all, avant filtrage météo)
mask_bldg_full = meta_all["bldg_id"].values == bldg_test
profile_test = days_raw_all[mask_bldg_full][0]  # à ajuster pour pointer exactement la même date que row_test

predicted = classify_new_day(profile_test, weather_test, bldg_id=bldg_test)
print(f"Cluster prédit : {predicted} | Cluster réel : {row_test['cluster_multi']}")

In [ ]:
def classify_new_days_batch(df_days, profile_col="profile", weather_cols=weather_feature_cols, bldg_col="bldg_id"):
    """
    df_days : DataFrame avec une colonne `profile_col` contenant des array(steps_per_day,),
              les colonnes météo, et éventuellement `bldg_col`.
    Retourne le DataFrame avec une colonne 'cluster_predicted' ajoutée.
    """
    predictions = []
    for _, row in df_days.iterrows():
        weather_vals = {c: row[c] for c in weather_cols}
        bldg = row[bldg_col] if bldg_col in df_days.columns else None
        pred = classify_new_day(row[profile_col], weather_vals, bldg_id=bldg)
        predictions.append(pred)

    df_out = df_days.copy()
    df_out["cluster_predicted"] = predictions
    return df_out

## Graphique final adapté aux daltoniens

Version accessible du graphique des profils de forme moyens par cluster : palette catégorielle validée
(distances ΔE vérifiées pour daltonisme protan/deutan/tritan, pas de rouge/vert adjacents) **combinée à
un style de trait différent par cluster** (l'identité de chaque cluster ne repose donc jamais sur la
couleur seule) et des étiquettes directes en encre neutre à l'extrémité de chaque courbe.

In [ ]:
# Palette catégorielle validée daltonisme (protan/deutan/tritan) — ordre fixe, ne jamais permuter
CB_PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
# Styles de trait : encodage redondant (l'identité ne dépend jamais de la couleur seule)
CB_LINESTYLES = ["-", "--", "-.", ":", (0, (3, 1, 1, 1)), (0, (5, 2)), (0, (1, 1)), (0, (4, 1, 1, 1, 1, 1))]

if k_multi_opt > len(CB_PALETTE):
    print(f"Attention : k={k_multi_opt} dépasse les {len(CB_PALETTE)} couleurs validées — "
          f"regrouper les clusters excédentaires ou étendre la palette avant de partager ce graphique.")

fig, ax = plt.subplots(figsize=(12, 7))
fig.patch.set_facecolor("#fcfcfb")
ax.set_facecolor("#fcfcfb")

for c in range(k_multi_opt):
    mask = cluster_labels_multi == c
    mean_profile = shape_ok[mask].mean(axis=0)
    color = CB_PALETTE[c % len(CB_PALETTE)]
    style = CB_LINESTYLES[c % len(CB_LINESTYLES)]

    ax.plot(
        HOURS, mean_profile, color=color, linestyle=style, linewidth=2.5,
        label=f"Cluster {c} (n={mask.sum()})",
    )
    # Étiquette directe à l'extrémité de la courbe, en encre neutre (pas de texte coloré)
    ax.annotate(
        f"C{c}", xy=(HOURS[-1], mean_profile[-1]), xytext=(6, 0),
        textcoords="offset points", va="center", fontsize=9,
        color="#0b0b0b", fontweight="bold",
    )

ax.set_xlabel("Heure")
ax.set_ylabel("Consommation normalisée (forme)")
ax.set_title(f"Profils de forme moyens par cluster (k={k_multi_opt}) — palette adaptée daltonisme")
ax.legend(loc="upper left", frameon=True, facecolor="#fcfcfb", edgecolor="#c3c2b7")
ax.grid(alpha=0.4, color="#e1e0d9")
for spine in ax.spines.values():
    spine.set_color("#c3c2b7")

plt.tight_layout()
plt.savefig(FIGURES / "cluster_shape_profiles_colorblind_safe.png", dpi=150, bbox_inches="tight")
plt.show()